# LLS：GT–LQ 退化统计与 GT-only 数据仿真

这个 notebook 分为两部分：

1. 说明怎样在数据集层面统计严格配对的 `GT ↔ LQ 中心帧` 退化；
2. 使用统计分布，把只有 GT 的图像合成为设备相关的 LQ。

目录中的 `GT.png / LQ.png / REF.png` 来自同一条 OPPO 样本 `1449_0`。`REF` 是模型的参考条件，不参与 GT–LQ 退化拟合，因为它来自不同拍摄时刻/视角，不能与 GT 做逐像素退化统计。

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv, io, json, random, sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
REPO_ROOT = next(p for p in candidates if (p / 'training' / 'synthetic_degradation.py').exists())
NOTEBOOK_DIR = REPO_ROOT / 'lls_degradation_notebook'
DATA_ROOT = Path('/nfs/wjl/datasets/lls')
STATS_DIR = REPO_ROOT / 'artifacts' / 'lls_degradation'
sys.path.insert(0, str(REPO_ROOT))

from training.lls_data import manifests, parent_id, read_triples
from training.synthetic_degradation import LLSSyntheticDegrader
from tools.analyze_lls_degradation import compute

key_stats = json.loads((NOTEBOOK_DIR / 'key_statistics.json').read_text())
print('repo:', REPO_ROOT)
print('example:', key_stats['example'])

## 示例三元组

LQ 是与 GT 内容对应的中心视频帧，尺寸更小；REF 是一个序列仅有一张的高质量参考照片。显示时把 LQ 放大到 GT 尺寸只是为了便于观察，不改变磁盘里的原生 LQ。

In [ ]:
gt = Image.open(NOTEBOOK_DIR / 'GT.png').convert('RGB')
real_lq = Image.open(NOTEBOOK_DIR / 'LQ.png').convert('RGB')
ref = Image.open(NOTEBOOK_DIR / 'REF.png').convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(
    axes, [gt, real_lq.resize(gt.size, Image.BICUBIC), ref],
    [f'GT {gt.size}', f'LQ {real_lq.size}（显示时放大）', f'REF {ref.size}']
):
    ax.imshow(image); ax.set_title(title); ax.axis('off')
plt.tight_layout()

# 前半部分：怎样统计整个数据集的 GT–LQ 退化

数据成员对应关系为：

- `gt.png`：中心时刻的高质量目标；
- `lq_sequence/04.png`：与 GT 对应的中心 LQ 帧；
- `ref.png`：独立参考照片，只用于条件注入。

统计流程是：原生 GT bicubic 缩到 LQ 尺寸 → ORB 特征与 RANSAC 相似变换做小幅配准 → 排除边界、过暗和过曝像素 → 拟合每通道 `LQ ≈ gain × GT + bias` → 计算亮度、色彩、边缘、拉普拉斯方差和高频残差代理 → 按设备汇总 `p10 / median / p90`。

高频残差是混合代理量，里面同时包含噪声、ISP、压缩、纹理、运动和配准误差，不能直接解释成传感器高斯噪声。

In [ ]:
# 这里读取全部 manifest，只做快速的数据清单统计，不解码 11.8 万张图。
rows = manifests(DATA_ROOT)
inventory = {}
for platform in ('OPPO', 'iPhone'):
    platform_rows = [r for r in rows if r['platform'] == platform]
    ids = [sample for r in platform_rows for sample in r['ids']]
    inventory[platform] = {
        'shards': len(platform_rows),
        'patches': len(ids),
        'parent_ids': len({parent_id(x) for x in ids}),
    }
inventory

### 原始图像扫描代码

下面的代码给出完整扫描入口。当前仓库附带的统计排除了图像预训练的 validation holdout，并采用每个平台 120 个不同 parent ID、每个 parent 随机一块，共 240 对，减少同一原图大量相邻 patch 带来的相关性。

- 保持 `RUN_RAW_AUDIT=False`：直接读取已经计算好的 `per_sample.csv`；
- 设为 `True` 且 `MAX_PARENTS_PER_PLATFORM=None`：覆盖所有 parent，每个 parent 一块；
- 再设 `ONE_PATCH_PER_PARENT=False`：逐一扫描全部 118,069 个 patch，耗时会明显增加。

In [ ]:
RUN_RAW_AUDIT = False
MAX_PARENTS_PER_PLATFORM = 120  # None 表示所有 parent
ONE_PATCH_PER_PARENT = True
AUDIT_SEED = 20261003

def build_selection(rows, max_parents=None, one_patch_per_parent=True, seed=20261003):
    rng = random.Random(seed)
    selected = defaultdict(dict)
    for platform in ('OPPO', 'iPhone'):
        items = [(r['tar'], sample) for r in rows if r['platform'] == platform for sample in r['ids']]
        if not one_patch_per_parent:
            chosen = items
        else:
            by_parent = defaultdict(list)
            for tar_path, sample in items:
                by_parent[parent_id(sample)].append((tar_path, sample))
            parents = sorted(by_parent)
            if max_parents is not None:
                parents = rng.sample(parents, min(max_parents, len(parents)))
            chosen = [rng.choice(by_parent[p]) for p in parents]
        for tar_path, sample in chosen:
            selected[tar_path][sample] = platform
    return selected

def scan_selected_pairs(selected):
    records = []
    for index, (tar_path, ids) in enumerate(sorted(selected.items()), 1):
        for sample, triple in read_triples(tar_path, set(ids)):
            records.append(compute(ids[sample], sample, triple))
        if index % 10 == 0:
            print(f'{index}/{len(selected)} shards, {len(records)} pairs')
    return records

if RUN_RAW_AUDIT:
    selection = build_selection(rows, MAX_PARENTS_PER_PLATFORM, ONE_PATCH_PER_PARENT, AUDIT_SEED)
    records = scan_selected_pairs(selection)
else:
    with (STATS_DIR / 'per_sample.csv').open(newline='') as f:
        records = list(csv.DictReader(f))

print('loaded pairs:', len(records))

### 指标定义

- `median_y_ratio`：有效像素上 `(Y_LQ+0.02)/(Y_GT+0.02)` 的中位数，反映曝光/亮度偏移；
- `luma_range_ratio`：LQ 与 GT 的亮度 `p99-p1` 之比，反映动态跨度压缩；
- `saturation_proxy_ratio`：平均 `max(R,G,B)-min(R,G,B)` 之比，作为色彩跨度代理；
- `rgb_gain_* / rgb_bias_*`：逐通道鲁棒仿射颜色拟合参数；
- `edge_strength_ratio`：GT 高梯度区域内 LQ/GT 梯度中位数之比；
- `laplacian_variance_ratio`：拉普拉斯方差之比，辅助描述清晰度；
- `hf_residual_sigma_proxy_8bit`：平坦区域高通残差的 MAD 尺度，单位为 8 位灰度级。

In [ ]:
KEY_METRICS = [
    'median_y_ratio', 'luma_range_ratio', 'saturation_proxy_ratio',
    'edge_strength_ratio', 'laplacian_variance_ratio',
    'hf_residual_sigma_proxy_8bit'
]

def number(row, key):
    value = row[key]
    return float(value) if value not in ('', None, 'None') else np.nan

audit_summary = {}
for platform in ('OPPO', 'iPhone'):
    subset = [r for r in records if r['platform'] == platform]
    audit_summary[platform] = {}
    for metric in KEY_METRICS:
        values = np.array([number(r, metric) for r in subset], dtype=float)
        values = values[np.isfinite(values)]
        audit_summary[platform][metric] = dict(zip(
            ('p10', 'median', 'p90'), np.quantile(values, (.1, .5, .9)).round(4)
        ))
audit_summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = {'OPPO': '#3478bf', 'iPhone': '#e07a2d'}
for ax, metric in zip(axes.ravel(), KEY_METRICS):
    for platform in ('OPPO', 'iPhone'):
        values = [number(r, metric) for r in records if r['platform'] == platform]
        ax.hist(values, bins=20, alpha=.55, label=platform, color=colors[platform])
    ax.set_title(metric); ax.grid(alpha=.2)
axes[0, 0].legend()
plt.tight_layout()

统计结果表明两种设备必须分开建模：OPPO 的 LQ 通常更暗、色彩跨度更小；iPhone 的 LQ 通常更亮。两者在缩到原生 LQ 尺寸后仍比 GT 更软。关键结果已固化在 `key_statistics.json`，完整逐样本数据位于项目的 `artifacts/lls_degradation/per_sample.csv`。

# 后半部分：只有 GT 时怎样仿真 LQ

仿真顺序为：

`GT → 抽取同平台的一整行联合参数 → GT 空间轻度高斯预模糊 → bicubic 到设备原生 LQ 尺寸 → RGB gain/bias → 亮度跨度与亮度中位数校准 → 色彩跨度校准 → 小幅信号相关扰动 → 8-bit RGB`

颜色、亮度、色彩和残差参数从同一条真实样本记录联合取得，保留它们之间的相关性。默认噪声只使用混合高频残差代理的 30%，避免把所有 ISP/配准差异错误地变成独立白噪声。

In [ ]:
PLATFORM = 'OPPO'
SEED = 42
NOISE_FRACTION = 0.3

degrader = LLSSyntheticDegrader(STATS_DIR, noise_fraction=NOISE_FRACTION)
synthetic_lq, degradation_params = degrader(
    gt, PLATFORM, np.random.default_rng(SEED), forbid_sample='1449_0'
)
synthetic_lq.save(NOTEBOOK_DIR / 'SYNTHETIC_LQ.png')
json.dumps(degradation_params.as_dict(), indent=2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 5))
display_images = [
    gt,
    real_lq.resize(gt.size, Image.BICUBIC),
    synthetic_lq.resize(gt.size, Image.BICUBIC),
    ref,
]
titles = ['GT', '真实 LQ（显示时放大）', '合成 LQ（显示时放大）', 'REF（只作条件）']
for ax, image, title in zip(axes, display_images, titles):
    ax.imshow(image); ax.set_title(title); ax.axis('off')
plt.tight_layout()

### 检查单个样本的指标

单张随机合成图不需要逐项等于真实 LQ；验收目标是大量合成图的分布接近真实数据分布。下面仅用于检查这一张示例是否落在合理范围。

In [ ]:
def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    return buffer.getvalue()

gt_blob = (NOTEBOOK_DIR / 'GT.png').read_bytes()
real_metrics = compute(PLATFORM, '1449_0_real', {'gt': gt_blob, 'lq': png_bytes(real_lq)})
synthetic_metrics = compute(PLATFORM, '1449_0_synthetic', {'gt': gt_blob, 'lq': png_bytes(synthetic_lq)})
comparison = {
    metric: {'real': round(real_metrics[metric], 4), 'synthetic': round(synthetic_metrics[metric], 4)}
    for metric in KEY_METRICS
}
comparison

### 批量生成模板

对一个 GT 目录批量生成时，应固定平台并为每张图片使用可复现但不同的随机种子。项目也提供命令行：

```bash
/opt/anaconda3/bin/python tools/simulate_lls_lq.py \
  --gt /path/to/gt_images --platform OPPO \
  --output /path/to/new_pairs --seed 42
```

训练时先在原生 LQ 尺寸生成，再把 LQ 放大到模型输入尺寸。不要直接在 512×512 上添加噪声来代替原生降采样链路。

## 边界

这套方法只能从 GT 合成单帧 LQ。它不能从一张 GT 得到真正独立的 REF，也不能产生具有遮挡、新显露区域和真实运动的多帧视频。原数据只有解码 PNG，因此当前不指定某个视频编码器或码率；如果之后拿到原视频，需要重新统计压缩伪影和时间一致性。